# Tahap 3 — Case Retrieval
## CBR Putusan Wanprestasi PN Surabaya

**Input:** `data/processed/cases.csv` → output Tahap 2

**Pendekatan:** TF-IDF + SVM (linear kernel)

**Split:** 80:20 (stratified)

**Output:**
- Fungsi `retrieve(query, k=5)` teruji
- `data/eval/queries.json` — 8 query uji + ground-truth
- `data/eval/retrieval_metrics.csv` — metrik evaluasi

| Cell | Fungsi |
|---|---|
| **Cell 1** | Import library |
| **Cell 2** | Load data & konfigurasi |
| **Cell 3** | Fungsi preprocessing query |
| **Cell 4** | TF-IDF Vectorizer |
| **Cell 5** | Split data 80:20 |
| **Cell 6** | Training & evaluasi SVM |
| **Cell 7** | Fungsi `retrieve()` & `retrieve_with_scores()` |
| **Cell 8** | Buat query uji & simpan `queries.json` |
| **Cell 9** | Evaluasi retrieval |
| **Cell 10** | Ringkasan akhir |

## Setup - Jangkar Working Directory ke Root Repository

**Jalankan cell ini SELALU sebagai cell pertama**, sebelum cell lain di notebook ini.

Notebook ini disimpan di dalam folder `notebooks/`, tapi seluruh path data di notebook ini
(mis. `'data/processed/cases.csv'`) ditulis **relatif terhadap root repository**, bukan terhadap
folder `notebooks/`. Secara default, Jupyter menjalankan kernel dengan *working directory* = folder
tempat file `.ipynb` itu sendiri berada. Tanpa cell ini, path seperti `'data/raw'` akan dicari di
`notebooks/data/raw` (tidak ada) dan menyebabkan `FileNotFoundError`.

Cell ini **aman dijalankan berkali-kali** (idempotent) — begitu working directory sudah berada di
root repository (folder yang punya subfolder `data/`), cell ini tidak akan berpindah lagi.

In [23]:
import os

# Jika folder 'data/' tidak ada di working directory saat ini, TAPI ada satu
# level di atasnya -> kita sedang di dalam notebooks/, pindah ke root repo.
if not os.path.isdir('data') and os.path.isdir(os.path.join('..', 'data')):
    os.chdir('..')

print('Working directory aktif :', os.getcwd())
print('Isi folder saat ini     :', sorted(os.listdir('.')))

assert os.path.isdir('data'), (
    "Folder 'data/' tidak ditemukan dari working directory ini.\n"
    "Pastikan struktur repo: <root>/notebooks/xx.ipynb dengan <root>/data/ di sebelahnya,\n"
    "dan notebook dibuka dari dalam struktur repo tersebut (bukan dipindah sendirian)."
)


Working directory aktif : c:\Users\LENOVO\Documents\Penalaran Komputer\Tugas_3\Tugas3.1
Isi folder saat ini     : ['.git', '.gitignore', 'README.md', 'data', 'logs', 'notebooks', 'requirements.txt']


## Cell 1 — Import Library

In [24]:
import os, re, json
import numpy as np
import pandas as pd
from typing import List
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)
from sklearn.metrics.pairwise import cosine_similarity

print('Semua library siap!')

Semua library siap!


## Cell 2 — Load Data & Konfigurasi

In [25]:
PATH_CASES_CSV    = 'data/processed/cases.csv'
PATH_QUERIES_JSON = 'data/eval/queries.json'
PATH_METRICS_CSV  = 'data/eval/retrieval_metrics.csv'

os.makedirs('data/eval', exist_ok=True)

# Load cases.csv
df = pd.read_csv(PATH_CASES_CSV, encoding='utf-8-sig')

# Hapus baris tanpa text_full
df = df[df['text_full'].notna() & (df['text_full'].str.len() > 100)].reset_index(drop=True)

# df_model: hanya label 0 dan 1 untuk training SVM
df_model = df[df['label_putusan'].isin([0, 1])].reset_index(drop=True)

print('=' * 62)
print('  KONFIGURASI TAHAP 3 — CASE RETRIEVAL')
print('=' * 62)
print(f'  Total kasus tersedia     : {len(df)}')
print(f'  Kasus untuk model (0/1)  : {len(df_model)}')
print(f'  Label Dikabulkan  (1)    : {(df_model["label_putusan"]==1).sum()}')
print(f'  Label Ditolak/NO  (0)    : {(df_model["label_putusan"]==0).sum()}')
print('=' * 62)

  KONFIGURASI TAHAP 3 — CASE RETRIEVAL
  Total kasus tersedia     : 54
  Kasus untuk model (0/1)  : 54
  Label Dikabulkan  (1)    : 30
  Label Ditolak/NO  (0)    : 24


## Cell 3 — Fungsi Preprocessing Query

In [26]:
STOPWORDS = {
    'dan', 'di', 'ke', 'dari', 'yang', 'ini', 'itu', 'atau',
    'pada', 'dengan', 'adalah', 'juga', 'oleh', 'serta', 'pula',
    'pun', 'nya', 'si', 'tidak', 'telah', 'akan', 'bahwa', 'untuk',
    'dalam', 'atas', 'tersebut', 'para', 'sebagai', 'sudah',
    'hal', 'maka', 'agar', 'karena', 'namun', 'tetapi', 'jika',
    'pihak', 'perkara', 'maupun', 'dapat', 'saat', 'telah', 'juga',
}

def preprocess_query(teks: str) -> str:
    """
    Preprocessing teks sebelum diubah ke vektor TF-IDF.
    Langkah:
    1) Lowercase
    2) Hapus tanda baca
    3) Normalisasi spasi
    4) Filter stopwords & kata pendek (< 3 karakter)
    """
    teks = teks.lower()
    teks = re.sub(r'[^\w\s]', ' ', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()
    tokens = [t for t in teks.split() if t not in STOPWORDS and len(t) >= 3]
    return ' '.join(tokens)

# Test
contoh = 'Penggugat menuntut ganti rugi karena tergugat wanprestasi tidak membayar'
print(f'Input  : {contoh}')
print(f'Output : {preprocess_query(contoh)}')
print()
print('Fungsi preprocess_query siap!')

Input  : Penggugat menuntut ganti rugi karena tergugat wanprestasi tidak membayar
Output : penggugat menuntut ganti rugi tergugat wanprestasi membayar

Fungsi preprocess_query siap!


## Cell 4 — TF-IDF Vectorizer

Membangun representasi vektor dokumen menggunakan `TfidfVectorizer` dari scikit-learn.
Seluruh corpus (58 dokumen) di-fit agar fungsi `retrieve()` bisa mencari di semua kasus.

In [27]:
# Preprocessing seluruh corpus
corpus   = df['text_full'].apply(preprocess_query).tolist()
case_ids = df['case_id'].tolist()

# Fit TF-IDF pada seluruh corpus
vectorizer = TfidfVectorizer(
    max_features = 5000,
    min_df       = 2,       # abaikan term yang hanya muncul di 1 dokumen
    max_df       = 0.95,    # abaikan term yang muncul di >95% dokumen
    ngram_range  = (1, 2),  # unigram + bigram
    sublinear_tf = True,    # log normalisasi TF
)

tfidf_matrix = vectorizer.fit_transform(corpus)

print('=' * 62)
print('  TF-IDF VECTORIZER')
print('=' * 62)
print(f'  Jumlah dokumen   : {tfidf_matrix.shape[0]}')
print(f'  Jumlah fitur     : {tfidf_matrix.shape[1]:,}')
print(f'  Contoh top terms : {vectorizer.get_feature_names_out()[:10].tolist()}')
print('=' * 62)
print('TF-IDF matrix siap!')

  TF-IDF VECTORIZER
  Jumlah dokumen   : 54
  Jumlah fitur     : 5,000
  Contoh top terms : ['000 biaya', '000 bukti', '000 delapan', '000 diberi', '000 dua', '000 empat', '000 enam', '000 jumlah', '000 lima', '000 limapuluh']
TF-IDF matrix siap!


## Cell 5 — Split Data Train/Test (80:20)

In [28]:
# Gunakan df_model (label 0 & 1) untuk training SVM
corpus_model   = df_model['text_full'].apply(preprocess_query).tolist()
y              = df_model['label_putusan'].tolist()
case_ids_model = df_model['case_id'].tolist()

# Transform menggunakan vectorizer yang sudah di-fit pada seluruh corpus
X = vectorizer.transform(corpus_model)

# Split 80:20 dengan stratify agar distribusi label seimbang di train & test
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, case_ids_model,
    test_size    = 0.2,
    random_state = 42,
    stratify     = y
)

print('=' * 62)
print('  SPLIT DATA 80:20')
print('=' * 62)
print(f'  Total data model   : {len(y)}')
print(f'  Data Train (80%)   : {X_train.shape[0]} dokumen')
print(f'  Data Test  (20%)   : {X_test.shape[0]} dokumen')
print(f'  Train — label 1    : {sum(y_train)} | label 0: {y_train.count(0)}')
print(f'  Test  — label 1    : {sum(y_test)}  | label 0: {y_test.count(0)}')
print('=' * 62)
print(f'  Case ID Test Set   : {ids_test}')

  SPLIT DATA 80:20
  Total data model   : 54
  Data Train (80%)   : 43 dokumen
  Data Test  (20%)   : 11 dokumen
  Train — label 1    : 24 | label 0: 19
  Test  — label 1    : 6  | label 0: 5
  Case ID Test Set   : ['case_024', 'case_047', 'case_013', 'case_044', 'case_029', 'case_037', 'case_041', 'case_036', 'case_021', 'case_014', 'case_051']


## Cell 6 — Training & Evaluasi Model SVM

In [29]:
# Training SVM dengan kernel linear (optimal untuk teks)
svm_model = SVC(
    kernel       = 'linear',
    C            = 1.0,
    probability  = True,
    random_state = 42
)
svm_model.fit(X_train, y_train)

# Prediksi pada test set
y_pred = svm_model.predict(X_test)

# Hitung metrik
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print('=' * 62)
print('  EVALUASI SVM — TEST SET (20%)')
print('=' * 62)
print(f'  Accuracy  : {acc:.4f}')
print(f'  Precision : {prec:.4f}  (weighted)')
print(f'  Recall    : {rec:.4f}  (weighted)')
print(f'  F1-Score  : {f1:.4f}  (weighted)')
print()
print('  Classification Report:')
print(classification_report(
    y_test, y_pred,
    target_names=['Ditolak/NO (0)', 'Dikabulkan (1)']
))
print('=' * 62)

  EVALUASI SVM — TEST SET (20%)
  Accuracy  : 0.5455
  Precision : 0.5303  (weighted)
  Recall    : 0.5455  (weighted)
  F1-Score  : 0.4935  (weighted)

  Classification Report:
                precision    recall  f1-score   support

Ditolak/NO (0)       0.50      0.20      0.29         5
Dikabulkan (1)       0.56      0.83      0.67         6

      accuracy                           0.55        11
     macro avg       0.53      0.52      0.48        11
  weighted avg       0.53      0.55      0.49        11



## Cell 7 — Fungsi `retrieve()`

Implementasi fungsi retrieval sesuai spesifikasi soal:
1. Pre-process query
2. Hitung vektor query (TF-IDF)
3. Hitung cosine-similarity dengan semua case vectors
4. Kembalikan top-k case_id

In [30]:
def retrieve(query: str, k: int = 5) -> List[str]:
    """
    Temukan k kasus paling mirip dengan query kasus baru.

    Args:
        query : str  — deskripsi singkat kasus baru
        k     : int  — jumlah kasus yang dikembalikan (default 5)

    Returns:
        List[str] — daftar case_id top-k paling mirip
    """
    # 1) Pre-process query
    query_bersih = preprocess_query(query)

    # 2) Hitung vektor query
    query_vec = vectorizer.transform([query_bersih])

    # 3) Hitung cosine-similarity dengan semua case vectors
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # 4) Kembalikan top-k case_id
    top_k_idx = np.argsort(sim_scores)[::-1][:k]
    return [case_ids[i] for i in top_k_idx]


def retrieve_with_scores(query: str, k: int = 5) -> List[dict]:
    """
    Versi retrieve() dengan detail skor similarity untuk analisis.

    Returns:
        List[dict] — setiap dict berisi case_id, no_perkara,
                     similarity, label, pihak
    """
    query_bersih = preprocess_query(query)
    query_vec    = vectorizer.transform([query_bersih])
    sim_scores   = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_k_idx    = np.argsort(sim_scores)[::-1][:k]

    hasil = []
    for i in top_k_idx:
        row = df.iloc[i]
        hasil.append({
            'case_id'    : case_ids[i],
            'no_perkara' : row['no_perkara'],
            'similarity' : round(float(sim_scores[i]), 4),
            'label'      : int(row['label_putusan']),
            'pihak'      : row['pihak'],
        })
    return hasil


# ── Test fungsi retrieve() ─────────────────────────────────────────────
print('Fungsi retrieve() siap!')
print()
query_test = 'tergugat tidak membayar hutang sesuai perjanjian wanprestasi ganti rugi'
hasil_test = retrieve_with_scores(query_test, k=5)

print(f'Query  : "{query_test}"')
print()
print(f'{"#":<3} {"case_id":<12} {"no_perkara":<26} {"similarity":<12} {"label"}')
print('-' * 70)
for i, r in enumerate(hasil_test, 1):
    lbl = {1: 'Dikabulkan', 0: 'Ditolak/NO', 2: 'Damai', -1: 'Unknown'}.get(r['label'], '?')
    print(f'{i:<3} {r["case_id"]:<12} {r["no_perkara"]:<26} {r["similarity"]:<12} {lbl}')

Fungsi retrieve() siap!

Query  : "tergugat tidak membayar hutang sesuai perjanjian wanprestasi ganti rugi"

#   case_id      no_perkara                 similarity   label
----------------------------------------------------------------------
1   case_038     13/Pdt.G/2025/PN Sby       0.0865       Ditolak/NO
2   case_019     563/Pdt.G/2025/PN Sby      0.0822       Dikabulkan
3   case_020     359/Pdt.G/2025/PN Sby      0.0646       Dikabulkan
4   case_051     1372/Pdt.G/2024/PN Sby     0.0524       Dikabulkan
5   case_003     75/Pdt.G/2025/PN Sby       0.0494       Dikabulkan


## Cell 8 — Buat Query Uji & Simpan `queries.json`

Siapkan 8 query uji beserta ground-truth case_id sesuai soal (minimal 5-10 query).

In [31]:
# ── Ground-truth berbasis kata kunci di teks (bukan dari retrieve()) ──────
# Cara: cari case_id yang teksnya mengandung kata kunci relevan per tema query
# Ini memastikan evaluasi objektif — ground-truth tidak berasal dari model

def cari_gt(keywords: list, label_filter: int = None, n: int = 3) -> list:
    """
    Cari case_id yang mengandung kata kunci di text_full.
    Args:
        keywords     : list kata kunci (OR — minimal 1 harus ada)
        label_filter : 0=Ditolak, 1=Dikabulkan, None=semua
        n            : jumlah case yang diambil
    Returns:
        List[str] — top-n case_id yang relevan
    """
    import re
    pola = '|'.join(keywords)
    mask = df['text_full'].str.contains(pola, case=False, na=False, regex=True)
    if label_filter is not None:
        mask = mask & (df['label_putusan'] == label_filter)
    hasil = df[mask]['case_id'].tolist()
    return hasil[:n]


queries_uji = [
    {
        'query_id'      : 'Q001',
        'query'         : 'tergugat wanprestasi tidak membayar hutang sesuai perjanjian kredit',
        'ground_truth'  : cari_gt(['hutang', 'kredit', 'pinjaman', 'mengembalikan'], label_filter=1, n=3),
        'expected_label': 1,
        'keterangan'    : 'Wanprestasi pembayaran hutang/kredit'
    },
    {
        'query_id'      : 'Q002',
        'query'         : 'gugatan penggugat ditolak eksepsi dikabulkan tidak dapat diterima',
        'ground_truth'  : cari_gt(['niet ontvankelijk', 'tidak dapat diterima', 'eksepsi diterima'], label_filter=0, n=3),
        'expected_label': 0,
        'keterangan'    : 'Gugatan ditolak / NO'
    },
    {
        'query_id'      : 'Q003',
        'query'         : 'perjanjian jual beli tanah bangunan tergugat ingkar janji tidak menyerahkan objek',
        'ground_truth'  : cari_gt(['jual beli', 'tanah', 'sertifikat', 'akta jual'], label_filter=1, n=3),
        'expected_label': 1,
        'keterangan'    : 'Wanprestasi jual beli properti'
    },
    {
        'query_id'      : 'Q004',
        'query'         : 'penggugat meminjamkan uang tergugat tidak mengembalikan pinjaman jatuh tempo',
        'ground_truth'  : cari_gt(['pinjaman', 'meminjam', 'jatuh tempo', 'pengembalian uang'], label_filter=1, n=3),
        'expected_label': 1,
        'keterangan'    : 'Wanprestasi pinjaman uang'
    },
    {
        'query_id'      : 'Q005',
        'query'         : 'kontrak kerja konstruksi bangunan tergugat tidak menyelesaikan pekerjaan',
        'ground_truth'  : cari_gt(['renovasi', 'konstruksi', 'pekerjaan bangunan', 'kontraktor'], label_filter=1, n=3),
        'expected_label': 1,
        'keterangan'    : 'Wanprestasi kontrak konstruksi'
    },
    {
        'query_id'      : 'Q006',
        'query'         : 'gugatan tidak dapat diterima kompetensi absolut pengadilan tidak berwenang',
        'ground_truth'  : cari_gt(['kompetensi absolut', 'tidak berwenang', 'kewenangan mengadili', 'sengketa kewenangan'], label_filter=0, n=3),
        'expected_label': 0,
        'keterangan'    : 'Ditolak karena kompetensi'
    },
    {
        'query_id'      : 'Q007',
        'query'         : 'tergugat cidera janji tidak membayar sewa menyewa ruangan kantor gedung',
        'ground_truth'  : cari_gt(['sewa menyewa', 'penyewa', 'uang sewa', 'kontrak sewa'], label_filter=1, n=3),
        'expected_label': 1,
        'keterangan'    : 'Wanprestasi sewa menyewa'
    },
    {
        'query_id'      : 'Q008',
        'query'         : 'pasal 1243 1320 1338 kuhperdata wanprestasi ganti rugi materiil immateriil',
        'ground_truth'  : cari_gt(['pasal 1243', 'ganti rugi materiil', 'immateriil'], label_filter=1, n=3),
        'expected_label': 1,
        'keterangan'    : 'Query berbasis pasal KUHPerdata'
    },
]

# Validasi: pastikan semua ground-truth tidak kosong
for q in queries_uji:
    if len(q['ground_truth']) == 0:
        print(f"PERINGATAN: {q['query_id']} ground-truth kosong! Perluas kata kunci.")

# Simpan ke data/eval/queries.json
with open(PATH_QUERIES_JSON, 'w', encoding='utf-8') as f:
    json.dump(queries_uji, f, ensure_ascii=False, indent=2)

print('=' * 62)
print(f'  QUERY UJI — {len(queries_uji)} QUERY TERSIMPAN')
print(f'  Path: {PATH_QUERIES_JSON}')
print(f'  Ground-truth: berbasis kata kunci di text_full (bukan retrieve)')
print('=' * 62)
for q in queries_uji:
    print(f"  {q['query_id']} | {q['keterangan']}")
    print(f"         Query        : {q['query'][:55]}...")
    print(f"         Ground-truth : {q['ground_truth']}")
    print()
print('queries.json tersimpan!')

  QUERY UJI — 8 QUERY TERSIMPAN
  Path: data/eval/queries.json
  Ground-truth: berbasis kata kunci di text_full (bukan retrieve)
  Q001 | Wanprestasi pembayaran hutang/kredit
         Query        : tergugat wanprestasi tidak membayar hutang sesuai perja...
         Ground-truth : ['case_001', 'case_003', 'case_005']

  Q002 | Gugatan ditolak / NO
         Query        : gugatan penggugat ditolak eksepsi dikabulkan tidak dapa...
         Ground-truth : ['case_002', 'case_004', 'case_007']

  Q003 | Wanprestasi jual beli properti
         Query        : perjanjian jual beli tanah bangunan tergugat ingkar jan...
         Ground-truth : ['case_001', 'case_003', 'case_005']

  Q004 | Wanprestasi pinjaman uang
         Query        : penggugat meminjamkan uang tergugat tidak mengembalikan...
         Ground-truth : ['case_005', 'case_006', 'case_009']

  Q005 | Wanprestasi kontrak konstruksi
         Query        : kontrak kerja konstruksi bangunan tergugat tidak menyel...
         Ground-t

## Cell 9 — Evaluasi Retrieval

Hitung Precision@k, Recall@k, dan F1@k untuk setiap query uji.

In [32]:
def eval_retrieval(queries: list, k: int = 5) -> pd.DataFrame:
    """
    Evaluasi fungsi retrieve() pada daftar query uji.
    Metrik: Precision@k, Recall@k, F1@k per query.

    Args:
        queries : list  — daftar query uji dari queries.json
        k       : int   — top-k yang dievaluasi

    Returns:
        pd.DataFrame — hasil evaluasi per query
    """
    rows = []
    for q in queries:
        gt        = set(q['ground_truth'])
        retrieved = set(retrieve(q['query'], k=k))

        tp        = len(gt & retrieved)
        precision = tp / len(retrieved) if retrieved else 0
        recall    = tp / len(gt)        if gt        else 0
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) > 0 else 0)

        rows.append({
            'query_id'   : q['query_id'],
            'keterangan' : q['keterangan'],
            'ground_truth': list(gt),
            'retrieved'  : list(retrieved),
            'tp'         : tp,
            'precision'  : round(precision, 4),
            'recall'     : round(recall,    4),
            'f1'         : round(f1,        4),
        })
    return pd.DataFrame(rows)


df_eval = eval_retrieval(queries_uji, k=5)

print('=' * 62)
print('  EVALUASI RETRIEVAL — top-5')
print('=' * 62)
print(f'{"Query":<6} {"Precision":<12} {"Recall":<10} {"F1":<10} {"TP/GT"}')
print('-' * 55)
for _, row in df_eval.iterrows():
    print(f'{row["query_id"]:<6} {row["precision"]:<12.4f} {row["recall"]:<10.4f} {row["f1"]:<10.4f} {row["tp"]}/{len(row["ground_truth"])}')
print('-' * 55)
print(f'{"RATA2":<6} {df_eval["precision"].mean():<12.4f} {df_eval["recall"].mean():<10.4f} {df_eval["f1"].mean():<10.4f}')
print('=' * 62)

# Simpan metrik
df_eval.to_csv(PATH_METRICS_CSV, index=False, encoding='utf-8-sig')
print(f'\nMetrik tersimpan: {PATH_METRICS_CSV}')

# Analisis kegagalan (Rejection Analysis)
print()
print('=' * 62)
print('  ANALISIS KEGAGALAN (Rejection Analysis)')
print('=' * 62)
gagal = df_eval[df_eval['f1'] < 1.0]
if len(gagal) == 0:
    print('  Semua query berhasil di-retrieve dengan sempurna.')
else:
    for _, row in gagal.iterrows():
        print(f"  {row['query_id']} — {row['keterangan']}")
        print(f"    F1={row['f1']:.4f} | TP={row['tp']}")
        missed = set(row['ground_truth']) - set(row['retrieved'])
        print(f"    Kasus tidak terambil: {missed}")
        print(f"    Kemungkinan: kosakata query terlalu umum atau dokumen pendek")
        print()
print('=' * 62)

  EVALUASI RETRIEVAL — top-5
Query  Precision    Recall     F1         TP/GT
-------------------------------------------------------
Q001   0.2000       0.3333     0.2500     1/3
Q002   0.2000       0.3333     0.2500     1/3
Q003   0.0000       0.0000     0.0000     0/3
Q004   0.0000       0.0000     0.0000     0/3
Q005   0.0000       0.0000     0.0000     0/3
Q006   0.2000       0.3333     0.2500     1/3
Q007   0.4000       0.6667     0.5000     2/3
Q008   0.2000       0.3333     0.2500     1/3
-------------------------------------------------------
RATA2  0.1500       0.2500     0.1875    

Metrik tersimpan: data/eval/retrieval_metrics.csv

  ANALISIS KEGAGALAN (Rejection Analysis)
  Q001 — Wanprestasi pembayaran hutang/kredit
    F1=0.2500 | TP=1
    Kasus tidak terambil: {'case_001', 'case_003'}
    Kemungkinan: kosakata query terlalu umum atau dokumen pendek

  Q002 — Gugatan ditolak / NO
    F1=0.2500 | TP=1
    Kasus tidak terambil: {'case_002', 'case_004'}
    Kemungkinan: kosa

## Cell 10 — Ringkasan Akhir Tahap 3

In [33]:
print()
print('=' * 62)
print('  RINGKASAN AKHIR TAHAP 3 — CASE RETRIEVAL')
print('=' * 62)
print(f'  Representasi Vektor : TF-IDF (unigram + bigram)')
print(f'  Model Retrieval     : SVM (linear kernel, C=1.0)')
print(f'  Split Data          : 80:20 (stratified)')
print(f'  Corpus              : {tfidf_matrix.shape[0]} dokumen × {tfidf_matrix.shape[1]:,} fitur')
print(f'  Train / Test        : {X_train.shape[0]} / {X_test.shape[0]} dokumen')
print()
print('  Performa SVM (classification test set):')
print(f'    Accuracy          : {acc:.4f}')
print(f'    Precision         : {prec:.4f}')
print(f'    Recall            : {rec:.4f}')
print(f'    F1-Score          : {f1:.4f}')
print()
print(f'  Performa Retrieval (cosine similarity, top-5, {len(queries_uji)} query):')
print(f'    Avg Precision@5   : {df_eval["precision"].mean():.4f}')
print(f'    Avg Recall@5      : {df_eval["recall"].mean():.4f}')
print(f'    Avg F1@5          : {df_eval["f1"].mean():.4f}')
print()
print('  Output Tahap 3:')
for path in [PATH_QUERIES_JSON, PATH_METRICS_CSV]:
    ada = '✅' if os.path.exists(path) else '❌'
    print(f'    {ada} {path}')
print(f'    ✅ Fungsi retrieve() teruji pada {len(queries_uji)} query')
print('=' * 62)
print('  STATUS: TAHAP 3 SELESAI! Lanjut ke Tahap 4 (Case Solution Reuse).')
print('=' * 62)


  RINGKASAN AKHIR TAHAP 3 — CASE RETRIEVAL
  Representasi Vektor : TF-IDF (unigram + bigram)
  Model Retrieval     : SVM (linear kernel, C=1.0)
  Split Data          : 80:20 (stratified)
  Corpus              : 54 dokumen × 5,000 fitur
  Train / Test        : 43 / 11 dokumen

  Performa SVM (classification test set):
    Accuracy          : 0.5455
    Precision         : 0.5303
    Recall            : 0.5455
    F1-Score          : 0.4935

  Performa Retrieval (cosine similarity, top-5, 8 query):
    Avg Precision@5   : 0.1500
    Avg Recall@5      : 0.2500
    Avg F1@5          : 0.1875

  Output Tahap 3:
    ✅ data/eval/queries.json
    ✅ data/eval/retrieval_metrics.csv
    ✅ Fungsi retrieve() teruji pada 8 query
  STATUS: TAHAP 3 SELESAI! Lanjut ke Tahap 4 (Case Solution Reuse).
